# Practical 5 - Training our own keypoint detector (part 1)

In this notebook, we will train a keypoint detector to do something like this:
![](https://i.imgur.com/iPQgiR2.png)

<div class="alert alert-block alert-warning">Note that you can run this notebook locally if you have GPU (and correctly installed Nvidia drivers) for training the keypoint detector. However, we will use [Google Colab](https://colab.google/) virtual machines (VMs)</div>

If you use colab, make sure to select a GPU or TPU accelator by changing the runtime type. TPUs are faster (do you know why?) but sometimes don't work properly, so if the TPU cannot be found, make sure to switch to a GPU.

## Keypoint Detector Basics

Before we start training the model, we will briefly go over the basics of the keypoint detector.

Recall, the goal is determine semantic locations (the keypoints) on an image. So we want to determine a set of points ${(x_i,y_i)}$ given an image $I$.

As usual in deep learning, we will train a model to do this. A model is simply a function that transforms an input (in our case the image) to a certain output (in our case a set of keypoints) and has a number of parameters (represented as $\theta$) : $\hat{y}= f_{\theta}(x)$.

To find a good set of parameters, we will use [gradient descent](https://en.wikipedia.org/wiki/Gradient_descent), which will iteratively update the parameters to bring the model outputs $\hat{y}$ closer to the desired output values $y$.

 To quantify what 'good' means for the parameters, we will need a loss function $\mathcal{L}$, that describes how close the model ouput $\hat{y}$ is to the desired output $y$. The loss values $\mathcal{L}(y,\hat{y})$ can then be used by the gradient-descent algorithm to find good values for the parameters $\theta $.

 For the keypoint detector, the input $x$ is our image $I$. The output are the keypoints, but instead of asking the model to predict a series of $(x,y)$ values, we make it predict 'heatmaps', which are monochrome images. For each type of semantic location on the image, a different heatmap is predicted and we call these heatmaps also the channels of the keypoint detector.
![example](https://imgur.com/WCQmJgv.png)

From these heatmaps we can then extract the (x,y) coordinates by searching for local maxima.









## Setup
<div class="alert alert-block alert-info"> <b>Task</b>: Uncomment and run the lines below to update airo-mono and to install the keypoint-detector and some dependencies.</div>



In [1]:
!pip install airo-typing==2025.4.0
!pip install airo-spatial-algebra==2025.4.0
!pip install airo-dataset-tools==2025.4.0

In [1]:
# sanity check; if this import fails, restart your runtime. (runtime -> restart session and start from this cell)
import airo_dataset_tools

In [5]:
%%capture
!git clone https://github.com/tlpss/keypoint-detection.git
# install kp detector. Need to navigate to folder first to avoid strange 'could not import' error
!cd keypoint-detection && pip install .

In [7]:
%%capture
# install font for visualization
!apt install fonts-freefont-ttf

# install the torch-xla package for TPU acceleration
# instructions from https://lightning.ai/docs/pytorch/stable/accelerators/tpu_basic.html
!pip install cloud-tpu-client==0.10 https://storage.googleapis.com/tpu-pytorch/wheels/colab/torch_xla-2.0-cp310-cp310-linux_x86_64.whl

<div class="alert alert-block alert-info"> <b>Task</b>: Log in to wandb with the account you created in the homework.</div>

In [ ]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 

## Gathering the data

In this section, we will set up the dataset(s).

We will use two datasets: the one you have collected in the homework and another one that we have created by collecting some images on the robot setup.


### Uploading your dataset


To do this:

- zip your dataset
- click `Files` in the taskbar on the left. Then press upload to upload your dataset (as a ZIP).
- unzip the file (use the linux `!unzip` command)
- rename your dataset root folder to `microplate-home` and the annotations file to `annotations.json`. You can use the linux `mv` command for this

<div class="alert alert-block alert-info"> <b>Task</b>: follow the steps above and use the cells below to set up your own dataset on the VM.</div>


In [2]:
#TODO: use unzip to unzip your dataset
!unzip --help

UnZip 6.00 of 20 April 2009, by Debian. Original by Info-ZIP.

Usage: unzip [-Z] [-opts[modifiers]] file[.zip] [list] [-x xlist] [-d exdir]
  Default action is to extract files in list, except those in xlist, to exdir;
  file[.zip] may be a wildcard.  -Z => ZipInfo mode ("unzip -Z" for usage).

  -p  extract files to pipe, no messages     -l  list files (short format)
  -f  freshen existing files, create none    -t  test compressed archive data
  -u  update files, create if necessary      -z  display archive comment only
  -v  list verbosely/show version info       -T  timestamp archive to latest
  -x  exclude files that follow (in xlist)   -d  extract files into exdir
modifiers:
  -n  never overwrite existing files         -q  quiet mode (-qq => quieter)
  -o  overwrite files WITHOUT prompting      -a  auto-convert any text files
  -j  junk paths (do not make directories)   -aa treat ALL files as text
  -U  use escapes for all non-ASCII Unicode  -UU ignore any Unicode fields
  -C  mat

In [ ]:
#TODO: use the mv command to rename your dataset folder and annotations file (if needed)
!mv --help

<div class="alert alert-block alert-info"> <b>Task</b>: Now run the cell below to download the second dataset</div>

In [5]:
# download the provided microplate datasetour
import os
if not os.path.exists('microplate-irm-setup'):
  !wget https://cloud.ilabt.imec.be/index.php/s/PY97wiWcB5Jw7gW/download/microplate_resized_1024x512.zip
  !unzip microplate_resized_1024x512.zip
  !mv microplate_resized_1024x512 microplate-irm-setup

<div class="alert alert-block alert-info"> <b>Task</b>: Run the cells below to verify if all datasets are in place. This will also show a few samples from the dataset we have provided. Do these look very different from yours?</div>


answer here

In [6]:
MICROPLATE_HOME_TRAIN_JSON = "microplate-home/annotations.json"
MICROPLATE_IRM_SETUP_TRAIN_JSON = "microplate-irm-setup/annotations_train.json"
MICROPLATE_IRM_SETUP_VAL_JSON = "microplate-irm-setup/annotations_val.json"
MICROPLATE_MERGED_TRAIN_JSON = "microplate-merged/annotations_train.json"

In [7]:
# check datasets are present
assert os.path.exists(MICROPLATE_IRM_SETUP_TRAIN_JSON), "did you forget to download the dataset we provide?"
assert os.path.exists(MICROPLATE_HOME_TRAIN_JSON),"did you forget to upload your own dataset?"

AssertionError: did you forget to upload your own dataset?

In [8]:

from airo_dataset_tools.coco_tools.fiftyone_viewer import view_coco_dataset
view_coco_dataset(MICROPLATE_IRM_SETUP_VAL_JSON,None,["keypoints"])

 100% |███████████████████| 24/24 [34.3ms elapsed, 0s remaining, 699.7 samples/s]     




███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.18.0

🤖  Ask the Docs Agent       https://docs.voxel51.com
🚀  Getting Started Guides   https://docs.voxel51.com/getting_started
💬  Join the Community       https://community.voxel51.com
🎓  Book a Workshop          https://voxel51.com/workshops

Notebook sessions cannot wait


We have now set up the datasets, and are ready to use them to train a keypoint detector!

## Training a first detector


<div class="alert alert-block alert-info"> <b>Task</b>: Run the cell below to start training your first detector! Depending on whether you have a GPU or TPU, this training run might take 10-20 minutes.</div>

<div class="alert alert-block alert-info"> <b>Task</b>: In the meantime, answer the questions in the cell below this command.</div>



In [9]:
!keypoint-detection train --json_dataset_path {MICROPLATE_HOME_TRAIN_JSON} --json_validation_dataset_path {MICROPLATE_IRM_SETUP_VAL_JSON} --json_test_dataset_path {MICROPLATE_IRM_SETUP_VAL_JSON}\
--keypoint_channel_configuration bottom_right:bottom_left:top_left:top_right --wandb_project irm-microplate-keypoints --max_epochs 150 --heatmap_sigma 3\
--early_stopping_relative_threshold -1.0 --precision 16 --devices 1 --maximal_gt_keypoint_pixel_distances "8" --backbone_type ConvNeXtUnet --batch_size 8\
--accelerator auto --devices 1 --augment_train --ap_epoch_freq 10 --learning_rate 3e-4

Traceback (most recent call last):
  File "/home/joon/miniconda3/envs/irm/bin/keypoint-detection", line 3, in <module>
    from keypoint_detection.tasks.cli import main
  File "/home/joon/miniconda3/envs/irm/lib/python3.10/site-packages/keypoint_detection/tasks/cli.py", line 4, in <module>
    from keypoint_detection.tasks.eval import eval_cli
  File "/home/joon/miniconda3/envs/irm/lib/python3.10/site-packages/keypoint_detection/tasks/eval.py", line 6, in <module>
    import pytorch_lightning as pl
  File "/home/joon/miniconda3/envs/irm/lib/python3.10/site-packages/pytorch_lightning/__init__.py", line 34, in <module>
    from lightning_fabric.utilities.seed import seed_everything  # noqa: E402
  File "/home/joon/miniconda3/envs/irm/lib/python3.10/site-packages/lightning_fabric/__init__.py", line 29, in <module>
    __import__("pkg_resources").declare_namespace(__name__)
ModuleNotFoundError: No module named 'pkg_resources'


<div class="alert alert-block alert-info"> <b>Task</b>: Answer the following questions while the model is training: Some questions are more generic, feel free to use the internet to answer those, for others you will need to take a look at the terminal output of the train command.</div>

- what is an epoch?
- what is a batch?
- what is the `batch size` we are using here?
- how many images are in the `train` dataset?
- how many images are in the `validation` dataset?
- how many images are in the `test`dataset?
- what is the difference between the train, validation and test dataset?



answer here

Among the first lines of output of the train command, is a link to a weights and biases run. It will look something like this:

 `wandb: 🚀 View run at https://wandb.ai/tlips/irm-microplate-keypoints/runs/<example>
`
**Task**: Click the link and explore all the different figures and graphs. Answer the questions below about the graphs:

- What is shown in each row of the `media/validation_XXX` figures?
- how are the train epoch_losses evolving over time? Is this a good or a bad sign?
- are the train/epoch_loss higher or lower than the val/epoch_loss values? Can you explain this?


## Second keypoint detector

Now we will train a second keypoint detector. This time we will mix the dataset you created in the homework with a part of the dataset we collected with the robot camera. The validation set will still be the same

<div class="alert alert-block alert-info"> <b>Task</b>: What do you think will happen to the train loss and validation loss compared to last time? Will it increase? Decrease?</div>

answer here

In [10]:
# create a merged dataset
!airo-dataset-tools merge-coco-datasets {MICROPLATE_HOME_TRAIN_JSON} {MICROPLATE_IRM_SETUP_TRAIN_JSON} --target-json-path {MICROPLATE_MERGED_TRAIN_JSON}

Usage: airo-dataset-tools merge-coco-datasets [OPTIONS] COCO_JSON_1
                                              COCO_JSON_2
Try 'airo-dataset-tools merge-coco-datasets --help' for help.

Error: Invalid value for 'COCO_JSON_1': Path 'microplate-home/annotations.json' does not exist.


<div class="alert alert-block alert-info"> <b>Task</b>: Take the command from last time. Change the train set to the newly created merged dataset. All other settings can be left to their previous values. Paste it in the cell below and run the training.</div>

In [11]:
#TODO: create a train command that uses the merged dataset as train set.

<div class="alert alert-block alert-info"> <b>Task</b>: Once this model is finished, take a look at the test/meanAP score. Is it higher or lower than for the previous model? The mAP should be around 90%!</div>



As a final task for this part of the practical, you need to obtain the link for the weights of the trained model. These will be used in the second part of the practical to grasp the microplate.

<div class="alert alert-block alert-info"> <b>Task</b>: To get this link, go the wandb run of the second model, then navigate to 'artifacts'. Press 'model' and copy the `full_name`, which looks similar to this one: `tlips/irm-microplate-keypoints/model-4o9lkyqg:v0`</div>


## Finished, go to part 2
Now you're done, head over to the other notebook (practical_5.2) to start working with the robot!